---
title: "Torque on a rotating sphere: a negative result"
subtitle: "The Stokes torque 8πμa³Ω is exact and the geometry never moves, which makes this the cleanest possible test of a resolved solver's torque. peclet.flow fails it — by 31%, structurally — and this page establishes exactly how."
author: "Peclet"
date: "2026-08-31"
categories: [flow, IBM, verification, analytic, torque, GPU]
jupyter: python3
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-engineering/peclet-examples/blob/main/examples/rotating-sphere-torque/index.ipynb){target="_blank"}
&nbsp;GPU example — the frozen page reads correctly without a solver.

## What you'll learn

Most of this gallery reports things that work. This one reports something that does not, because the
measurement is unambiguous and the consequence matters: **`peclet.flow` can compute the force on a
resolved body to round-off, and cannot yet compute the torque.**

A sphere spinning at angular velocity $\Omega$ in unbounded quiescent Stokes flow feels

$$
\mathbf{T} = -8\pi\mu a^{3}\,\boldsymbol\Omega ,
$$ {#eq-stokes-torque}

exactly. The test is unusually clean for three reasons: the answer is closed-form with no series and
no fitted constant; a sphere is invariant under rotation about its own axis, so **the geometry never
moves** — no fresh cells, no rebuild, none of the [moving-boundary
machinery](../moving-sphere-drag/index.qmd) is in play; and the net *force* is zero by symmetry, so
the same run checks that too.

## The two ways the solver can report a torque

Both come from the same design note, and this page is the first thing to put a number on either.

- **The reconstructed traction**, `hydro_force_torque` — integrates $(-p\mathbf I + \mu(\nabla
  \mathbf u + \nabla\mathbf u^{\mathsf T}))$ against the exact aperture wall-area vector. Its
  *force* is known to under-read by a resolution-independent ~29%, which is why it is kept as a
  diagnostic only.
- **The discrete reaction**, `hydro_force_torque_reaction` — takes the force from the momentum the
  fluid actually lost. Per fluid cell $R_i = -\nabla\pi_i + F_{\text{wall},i}$; summed over a body's
  owner region the $\nabla\pi$ telescopes to the region boundary, which for a single body in a
  periodic box is identically zero. **That is why the force is exact** — gated to $-8.8\times
  10^{-15}$ elsewhere in this suite. The torque takes the *first moment* of the same quantity, and
  $\sum \mathbf r \times \nabla\pi$ over a periodic box is **not** zero, because $\mathbf r$ is not
  periodic. Nothing guarantees the torque.

In [ ]:
#| label: bootstrap
#| code-summary: "Environment bootstrap (installs peclet from PyPI on Colab/Binder)"
import importlib.util, os, subprocess, sys
_local = os.environ.get("PECLET_LOCAL_BUILD")
if _local:
    for p in _local.split(os.pathsep):
        sys.path.insert(0, p)
elif importlib.util.find_spec("peclet") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peclet"], check=True)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from peclet import flow as sdflow

plt.rcParams.update({"figure.dpi": 130, "font.size": 9, "axes.axisbelow": True,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})
RHO, MU = 1.0, 0.1
NU = MU / RHO
KN_R, KI_I, KI_R = 16, 2, 17
OMEGA = 1e-3                       # rotational Reynolds number rho*Omega*a^2/mu is then O(1)

## The setup

One sphere at the centre of a periodic box, spinning about $z$. `set_instance_motion` carries the
angular velocity; `set_solid_from_scene` builds the geometry once and it is never rebuilt, because
a rotating sphere does not change shape.

In [ ]:
#| label: driver
def run(N, R, steps=1000, dt=20.0):
    x0 = 0.5 * N
    node_ints = np.array([1, -1, -1], dtype=np.int32)
    node_reals = np.zeros(KN_R); node_reals[0] = R
    node_reals[14] = 1.0; node_reals[15] = 1.0
    inst_ints = np.zeros((1, KI_I), dtype=np.int32); inst_ints[0] = (0, -1)
    inst_reals = np.zeros((1, KI_R))
    inst_reals[0, 0:3] = (x0, x0, x0)
    inst_reals[0, 6] = 1.0; inst_reals[0, 7] = 1.0

    s = sdflow.Solver(N, N, N)
    s.set_rho(RHO); s.set_mu(MU); s.set_dt(dt)
    s.set_advection(False)                       # Stokes: @eq-stokes-torque is the exact answer
    s.set_velocity_solver_params(100); s.set_pressure_solver_params(25)
    s.set_pressure_multigrid(True, levels=4)
    s.set_scene(node_ints, node_reals, inst_ints.ravel(), inst_reals.ravel(), periodic=True)
    s.set_instance_motion(0, lin_vel=[0.0, 0.0, 0.0], ang_vel=[0.0, 0.0, OMEGA],
                          center=[x0, x0, x0])
    s.set_solid_from_scene(True)

    t0 = time.time()
    for _ in range(steps):
        s.step()
    reaction = np.asarray(s.hydro_force_torque_reaction())
    traction = np.asarray(s.hydro_force_torque())
    exact = 8 * np.pi * MU * R ** 3 * OMEGA
    return dict(Texact=exact, Treact=abs(reaction[1][0][2]), Ttrac=abs(traction[1][0][2]),
                Fnet=np.linalg.norm(reaction[0][0]) / (MU * R ** 2 * OMEGA),
                c=(4 / 3) * np.pi * R ** 3 / N ** 3, N=N, R=R, wall=time.time() - t0)

## The measurement

Two ladders. Growing the **box** at fixed sphere resolution isolates the periodic-array effect;
refining the **grid** at fixed solid fraction isolates discretisation. If the deficit were either
of those, one of the two would move it.

In [ ]:
#| label: ladders
print("  N    R/h    c        T_exact       T_reaction     err      T_traction     err     |F|net")
rows = []
for N, R in ((64, 9.6), (96, 9.6), (128, 9.6), (96, 14.4)):
    r = run(N, R)
    rows.append(r)
    print("  %3d  %5.2f  %.2e  %.6e  %.6e  %+7.2f%%  %.6e  %+7.2f%%  %.1e"
          % (N, R, r["c"], r["Texact"], r["Treact"], 100 * (r["Treact"] / r["Texact"] - 1),
             r["Ttrac"], 100 * (r["Ttrac"] / r["Texact"] - 1), r["Fnet"]))

The reaction column is flat: a factor of eight in box volume, 3.4 in solid fraction and two in
$R/h$ move it by less than a percentage point. The traction column is not flat — and the reason is
not resolution at all.

## Step 2 — The traction torque never settles

Run one case and watch, rather than taking a single reading at an arbitrary step count.

In [ ]:
#| label: trace
N, R = 96, 14.4
x0 = 0.5 * N
node_ints = np.array([1, -1, -1], dtype=np.int32)
node_reals = np.zeros(KN_R); node_reals[0] = R
node_reals[14] = 1.0; node_reals[15] = 1.0
inst_ints = np.zeros((1, KI_I), dtype=np.int32); inst_ints[0] = (0, -1)
inst_reals = np.zeros((1, KI_R)); inst_reals[0, 0:3] = (x0, x0, x0)
inst_reals[0, 6] = 1.0; inst_reals[0, 7] = 1.0
s = sdflow.Solver(N, N, N)
s.set_rho(RHO); s.set_mu(MU); s.set_dt(20.0); s.set_advection(False)
s.set_velocity_solver_params(100); s.set_pressure_solver_params(25)
s.set_pressure_multigrid(True, levels=4)
s.set_scene(node_ints, node_reals, inst_ints.ravel(), inst_reals.ravel(), periodic=True)
s.set_instance_motion(0, lin_vel=[0, 0, 0], ang_vel=[0, 0, OMEGA], center=[x0, x0, x0])
s.set_solid_from_scene(True)

# split the pressure into the fluid and the solid interior
g = np.arange(N).astype(float)
X, Y, Z = np.meshgrid(g, g, g, indexing="ij")
inside = (np.sqrt((X - x0) ** 2 + (Y - x0) ** 2 + (Z - x0) ** 2) - R < 0).ravel()

Texact = 8 * np.pi * MU * R ** 3 * OMEGA
trace = []
print("  step |  reaction err |  traction err | max|u|     std(P) FLUID   std(P) SOLID")
for k in range(1, 9):
    for _ in range(200):
        s.step()
    react = abs(np.asarray(s.hydro_force_torque_reaction())[1][0][2])
    trac = abs(np.asarray(s.hydro_force_torque())[1][0][2])
    u = np.asarray(s.get_u()); P = np.asarray(s.get_p()).ravel()
    trace.append((200 * k, react / Texact - 1, trac / Texact - 1,
                  np.abs(u).max(), P[~inside].std(), P[inside].std()))
    print("  %4d |   %+7.2f%%    |   %+7.2f%%    | %.4e  %.4e     %.4e"
          % (200 * k, 100 * trace[-1][1], 100 * trace[-1][2], trace[-1][3], trace[-1][4],
             trace[-1][5]))

The velocity field is **steady to four digits from step 200 onward**, and the reaction torque
settles onto its −31% within a few hundred steps and then holds to six digits. The traction torque
does neither: it walks steadily upward, crossing the exact answer on its way past.

The last two columns say why. The pressure **in the fluid** is constant — it is also tiny, as it
should be, since the physical pressure field of a rotating sphere in Stokes flow is uniform. The
pressure **inside the solid** grows linearly with step count, and the traction torque tracks it.
Those cells are decoupled from the fluid by the cut-cell pressure operator, so nothing constrains
their value; the incremental scheme keeps accumulating into them, and the surface integral reads
them.

In [ ]:
#| label: fig-trace
#| fig-cap: "Left: the two torque routes against step count on a velocity field that stopped changing at step 200. Right: the pressure spread, split between the fluid and the solid interior — the fluid is flat, the solid drifts, and the traction torque follows the solid."
#| code-fold: true
tt = np.array([t[0] for t in trace])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.2, 2.9))
a1.plot(tt, [100 * t[1] for t in trace], "o-", color="#4c72b0", label="discrete reaction")
a1.plot(tt, [100 * t[2] for t in trace], "s-", color="#c44e52", label="reconstructed traction")
a1.axhline(0, color="0.3", ls="--", lw=1.0)
a1.set_xlabel("steps"); a1.set_ylabel("error in $T_z$ [%]")
a1.legend(fontsize=7.5, frameon=False); a1.grid(alpha=0.3)
a2.plot(tt, [t[4] for t in trace], "o-", color="#4c72b0", label="fluid")
a2.plot(tt, [t[5] for t in trace], "s-", color="#c44e52", label="solid interior")
a2.set_yscale("log"); a2.set_xlabel("steps"); a2.set_ylabel("std$(P)$")
a2.legend(fontsize=7.5, frameon=False); a2.grid(which="both", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
#| label: fig-ladders
#| fig-cap: "Relative error in the computed torque. The reaction route sits at −31% and does not care about the box or the grid; the traction route scatters over forty percentage points. The dashed line is the exact answer."
#| code-fold: true
fig, ax = plt.subplots(figsize=(6.0, 3.0))
lab = ["$N$=%d, $R/h$=%.1f" % (r["N"], r["R"]) for r in rows]
x = np.arange(len(rows))
ax.bar(x - 0.18, [100 * (r["Treact"] / r["Texact"] - 1) for r in rows], 0.34,
       color="#4c72b0", label="discrete reaction")
ax.bar(x + 0.18, [100 * (r["Ttrac"] / r["Texact"] - 1) for r in rows], 0.34,
       color="#c44e52", label="reconstructed traction")
ax.axhline(0, color="0.3", lw=1.0, ls="--")
ax.set_xticks(x); ax.set_xticklabels(lab, fontsize=7.5)
ax.set_ylabel("error in $T_z$ vs $8\\pi\\mu a^3\\Omega$ [%]")
ax.legend(fontsize=8, frameon=False); ax.grid(axis="y", alpha=0.3)
plt.show()

## Step 3 — What the ladders rule out

**Not the periodic box.** The first three rows hold the sphere at $R/h = 9.6$ and grow the box, a
factor of eight in volume and 3.4 in solid fraction. The reaction error moves by less than a
percentage point. A periodic-array correction must vanish as $c \to 0$; this does not.

**Not the grid.** The last row doubles $R/h$ at fixed $c$. Same answer.

**Not the time step or convergence.** The reaction torque is unchanged from 600 to 4000 steps to
six digits, on a velocity field that stopped moving long before.

**Not the force.** The net force comes out at the level of $10^{-13}$ of $\mu R^2\Omega$, which is
what symmetry demands — the same run that gets the torque wrong gets the force exactly right.

So there are **two independent defects**, and both now have a mechanism.

## Step 4 — Why

**The traction torque reads unconstrained data.** The cut-cell pressure operator decouples cells
whose centre lies inside the solid; nothing pins their value, and the incremental scheme accumulates
into them without bound. The surface integral samples pressure at cut cells, some of which are
solid-centred, so it picks that drift up. The fluid solution is untouched — which is why every
*drag* result in this gallery is unaffected, and why this shows up only in a case whose physical
pressure is uniform and therefore has nothing to mask it.

**The reaction torque is missing the transposed velocity gradient.** The viscous operator here is
the Laplacian $\nabla\cdot(\mu\nabla\mathbf u)$, not the full stress divergence
$\nabla\cdot\!\left[\mu(\nabla\mathbf u + \nabla\mathbf u^{\mathsf T})\right]$. For constant
$\mu$ and a divergence-free field the two agree in the interior, because
$\nabla\cdot(\nabla\mathbf u^{\mathsf T}) = \nabla(\nabla\cdot\mathbf u) = 0$ — but they do **not**
agree on a *boundary traction*, and the reaction budget's wall term is exactly a boundary traction.
For a rotating sphere the transposed part carries exactly a third of the torque, so a route built
on the Laplacian alone recovers exactly two thirds of it.

**And the third is exact.** For the isolated Stokes rotlet $\mathbf u = (\mathbf A\times\mathbf
r)/r^3$ with $\mathbf A = a^3\boldsymbol\Omega$, on the surface $|\mathbf x| = a$ with
$\mathbf n = \mathbf x/a$,

$$
(\nabla\mathbf u)\cdot\mathbf n = -\frac{2(\mathbf A\times\mathbf x)}{a^{4}},
\qquad
(\nabla\mathbf u^{\mathsf T})\cdot\mathbf n = -\frac{(\mathbf A\times\mathbf x)}{a^{4}} .
$$ {#eq-transpose}

The transposed part is exactly **half** of the plain gradient *pointwise on the surface*, so it is
one third of their sum before any integration and no cancellation can rescue it. With uniform
pressure and $\oint \mathbf x\times(\mathbf A\times\mathbf x)\,\mathrm dS = \frac{8\pi}{3}a^4\mathbf
A$, the full traction gives $-8\pi\mu a^3\Omega$ and the Laplacian-only traction gives exactly
$-\frac{16}{3}\pi\mu a^3\Omega$ — **two thirds**. The prediction is therefore $-33.33\%$, and we
measure $-31.0\%$; the couple of points between them are discretisation, and the deficit deepens
toward the prediction as the box grows.

**Why a solver can carry this invisibly.** The transposed term integrates to **zero in the net
force** — Gauss's theorem plus continuity — but it does not cancel under the $\mathbf r\times$
weighting of the torque. A code validated on drag alone will never see it. That is the single best
argument for making the rotating sphere a standing gate.

This is not a novel observation. Maitri et al. [-@maitri2018] ran precisely this a-priori test —
prescribe the exact rotlet, integrate the surface stress, compare to $8\pi\mu a^3\Omega$ — on the
earlier immersed-boundary method of Deen et al. [-@deen2012], which omits the same term, and
measured **34.34 / 33.04 / 33.24 / 33.32%** at $a/h = 5, 10, 20, 40$: a resolution-independent
plateau, with the stated cause that *"the ignored transposed terms contribute 1/3 of the total
analytical torque value"*. Our −31.0% is the same plateau in a different code.

It also explains why the obvious repair fails. Reading $R_i = -\nabla\pi_i + F_{\text{wall},i}$, one
might add $\sum\mathbf r\times\nabla\pi$ back with the momentum operator's own staggered pressure
gradient. That was tried: it moves the error from −31% to **−84%**. The pressure moment is not the
missing piece, because the missing piece is in the *viscous* term.

## Consequences, stated plainly

- `ResolvedCfdDem`'s `apply_torque` stays **off by default**. That was already the case on the
  grounds that the torque was *unvalidated*; it is now on the grounds that it is *wrong*, by a
  known mechanism, by about a third.
- A freely rotating resolved grain is not yet supported by this solver. Anything that depends on
  the hydrodynamic torque — a Jeffery orbit, a rotating non-spherical settler, torque-driven
  segregation — is blocked until this is fixed, and would silently be 31% off if attempted.
- The **force** is unaffected. Everything the resolved CFD-DEM loop currently does rests on the
  force, which is exact by an argument this measurement does not touch.

## Adapt this yourself

- **Use this as the gate.** @eq-stokes-torque has no fitted constant, the geometry is static and the
  run is a couple of minutes. Any candidate torque implementation should be held to it before
  anything else — and the a-priori version is cheaper still: prescribe the exact rotlet, integrate
  the stress, and skip the flow solve entirely, which is how Maitri et al. isolated the same term.
- **Then go beyond Stokes.** With a working torque, the natural next references are the rotary
  oscillation (exact, the rotational analogue of the [oscillating
  sphere](../oscillating-sphere/index.qmd)) and the finite-Reynolds-number torque of a steadily
  rotating sphere.
- **Check your own solver.** The recipe transfers: spin a sphere, take the torque, compare to
  $8\pi\mu a^3\Omega$, and refine the box and the grid separately. Two ladders and an exact answer
  is all it takes to tell a bias from a bug.

## Reproduce this

```bash
PECLET_LOCAL_BUILD=/path/to/suite/flow/build_l3_cuda \
  quarto render examples/rotating-sphere-torque/index.qmd --execute
```